## Importing the data

In [20]:
import yfinance as yf
import datetime as dt

# Set parameters for the data pull
ticker = "^GSPC"
end_period = dt.date.today()
beg_period = end_period - dt.timedelta(days=30*365) 

sp500_data = yf.download(ticker, start=beg_period, end=end_period)

print(sp500_data)

C:\Users\nisha\AppData\Local\Temp\ipykernel_29536\1288523646.py:9: FutureWarning: YF.download() has changed argument auto_adjust default to True
  sp500_data = yf.download(ticker, start=beg_period, end=end_period)
[*********************100%***********************]  1 of 1 completed

Price             Close         High          Low         Open      Volume
Ticker            ^GSPC        ^GSPC        ^GSPC        ^GSPC       ^GSPC
Date                                                                      
1996-04-29   654.159973   654.710022   651.599976   653.460022   344030000
1996-04-30   654.169983   654.590027   651.049988   654.159973   393390000
1996-05-01   654.580017   656.440002   652.260010   654.169983   404620000
1996-05-02   643.380005   654.580017   642.130005   654.580017   442960000
1996-05-03   641.630005   648.450012   640.229980   643.380005   434010000
...                 ...          ...          ...          ...         ...
2026-04-15  7022.950195  7026.240234  6967.129883  6978.169922  5278610000
2026-04-16  7041.279785  7051.229980  7008.520020  7037.779785  5173650000
2026-04-17  7126.060059  7147.520020  7074.549805  7074.549805  6145300000
2026-04-20  7109.140137  7122.649902  7084.410156  7117.049805  4661130000
2026-04-21  7064.009766  

## Adding Features

### Daily Returns

In [21]:
import numpy as np

#Taking log returns
sp500_data["Return"] = np.log(sp500_data["Close"] / sp500_data["Close"].shift(1)) * 100

#remove NaN rows
sp500_data.dropna(inplace=True)

### Rolling GARCH (1,1)

In [22]:
from arch import arch_model
import numpy as np

# Inputs
window_size = 1260  
test_start = window_size
total_days = len(sp500_data)

# Store results
rolling_volatility = []
current_params = {}
current_variance = 0

# The Loop
for i in range(test_start, total_days):
    
    # RE-FIT (Every 30 days)
    if (i - test_start) % 30 == 0:
        # Use the last 5 years of data to tune the model
        train_window = sp500_data["Return"].iloc[i-window_size : i]
        
        # Configure and fit GARCH(1,1)
        model = arch_model(train_window, mean='Zero', vol='GARCH', p=1, q=1, dist='t')
        res = model.fit(disp='off')
        
        # Save new parameters - NOTE THE INDEXING!
        current_params = {
            'omega': res.params['omega'],
            'alpha': res.params['alpha[1]'],  
            'beta':  res.params['beta[1]']     
        }
        
        # Reset current variance to the model's last estimate
        current_variance = res.conditional_volatility.iloc[-1]**2

    # MANUAL RECURSION (Daily Update)
    yesterday_return = sp500_data["Return"].iloc[i-1]
    
    # Calculate Variance for Today using GARCH formula
    next_variance = (current_params['omega'] + 
                     current_params['alpha'] * (yesterday_return**2) + 
                     current_params['beta'] * current_variance)
    
    # Store Volatility (Standard Deviation)
    rolling_volatility.append(np.sqrt(next_variance))
    
    # Update variance for tomorrow's loop
    current_variance = next_variance

# Initialize the column and assign results cleanly using pandas .loc
sp500_data['Rolling_GARCH_Vol'] = np.nan
sp500_data.loc[sp500_data.index[test_start:], 'Rolling_GARCH_Vol'] = rolling_volatility

print(sp500_data)

Price             Close         High          Low         Open      Volume  \
Ticker            ^GSPC        ^GSPC        ^GSPC        ^GSPC       ^GSPC   
Date                                                                         
1996-04-30   654.169983   654.590027   651.049988   654.159973   393390000   
1996-05-01   654.580017   656.440002   652.260010   654.169983   404620000   
1996-05-02   643.380005   654.580017   642.130005   654.580017   442960000   
1996-05-03   641.630005   648.450012   640.229980   643.380005   434010000   
1996-05-06   640.809998   644.640015   636.190002   641.630005   375820000   
...                 ...          ...          ...          ...         ...   
2026-04-15  7022.950195  7026.240234  6967.129883  6978.169922  5278610000   
2026-04-16  7041.279785  7051.229980  7008.520020  7037.779785  5173650000   
2026-04-17  7126.060059  7147.520020  7074.549805  7074.549805  6145300000   
2026-04-20  7109.140137  7122.649902  7084.410156  7117.049805  

### Risk Metric Vol

In [23]:
# 1. Set parameters (Standard RiskMetrics lambda for daily data = 0.94)
lambda_param = 0.94
ewm_alpha = 1 - lambda_param

# 2. Calculate daily variance (squared returns)
squared_returns = sp500_data["Return"]**2

# 3. Calculate EWMA Variance and shift by 1 to prevent data leakage
# adjust=False mimics the exact RiskMetrics recursive definition
conditional_variance_rm = squared_returns.ewm(alpha=ewm_alpha, adjust=False).mean().shift(1)

# 4. Take the square root to convert variance to standard deviation (Volatility)
sp500_data['RiskMetrics_Vol'] = np.sqrt(conditional_variance_rm)

print(sp500_data)

Price             Close         High          Low         Open      Volume  \
Ticker            ^GSPC        ^GSPC        ^GSPC        ^GSPC       ^GSPC   
Date                                                                         
1996-04-30   654.169983   654.590027   651.049988   654.159973   393390000   
1996-05-01   654.580017   656.440002   652.260010   654.169983   404620000   
1996-05-02   643.380005   654.580017   642.130005   654.580017   442960000   
1996-05-03   641.630005   648.450012   640.229980   643.380005   434010000   
1996-05-06   640.809998   644.640015   636.190002   641.630005   375820000   
...                 ...          ...          ...          ...         ...   
2026-04-15  7022.950195  7026.240234  6967.129883  6978.169922  5278610000   
2026-04-16  7041.279785  7051.229980  7008.520020  7037.779785  5173650000   
2026-04-17  7126.060059  7147.520020  7074.549805  7074.549805  6145300000   
2026-04-20  7109.140137  7122.649902  7084.410156  7117.049805  

### Adding VIX

In [24]:
import yfinance as yf
import numpy as np
import pandas as pd

# Download VIX data (utilizing beg_period and end_period from earlier)
vix_data = yf.download("^VIX", start=beg_period, end=end_period, progress=False)

# Safely extract 'Close' depending on how yfinance formats the current output
if isinstance(vix_data.columns, pd.MultiIndex):
    vix_series = vix_data['Close', '^VIX']
else:
    vix_series = vix_data['Close']

# Rename for a clean join
vix_series.name = 'US_VIX'

# Flatten the S&P 500 columns if they are MultiIndex (drops the '^GSPC' row)
if isinstance(sp500_data.columns, pd.MultiIndex):
    sp500_data.columns = sp500_data.columns.droplevel(1)
# Join to our main dataframe
sp500_data = sp500_data.join(vix_series, how='left')


# Calculate Daily Implied Volatility AND shift by 1 to prevent data leakage
sp500_data['VIX_Daily_Shifted'] = (sp500_data['US_VIX'] / np.sqrt(252)).shift(1)

# Clean up the raw column and drop the new NaN created by the shift
sp500_data.drop(columns=['US_VIX'], inplace=True)
sp500_data.dropna(inplace=True)

print(sp500_data)

                  Close         High          Low         Open      Volume  \
Date                                                                         
2001-04-27  1253.050049  1253.069946  1234.520020  1234.520020  1091300000   
2001-04-30  1249.459961  1269.300049  1243.989990  1253.050049  1266800000   
2001-05-01  1266.439941  1266.469971  1243.550049  1249.459961  1181300000   
2001-05-02  1267.430054  1272.930054  1257.699951  1266.439941  1342200000   
2001-05-03  1248.579956  1267.430054  1239.880005  1267.430054  1137900000   
...                 ...          ...          ...          ...         ...   
2026-04-15  7022.950195  7026.240234  6967.129883  6978.169922  5278610000   
2026-04-16  7041.279785  7051.229980  7008.520020  7037.779785  5173650000   
2026-04-17  7126.060059  7147.520020  7074.549805  7074.549805  6145300000   
2026-04-20  7109.140137  7122.649902  7084.410156  7117.049805  4661130000   
2026-04-21  7064.009766  7137.270020  7050.200195  7122.640137  

C:\Users\nisha\AppData\Local\Temp\ipykernel_29536\915633256.py:6: FutureWarning: YF.download() has changed argument auto_adjust default to True
  vix_data = yf.download("^VIX", start=beg_period, end=end_period, progress=False)


### Adding RSI (Momentum Indicator)

In [25]:
import os
# (Assuming your other imports from earlier are still active)

# Re-introduce the export path from your very first snippet
path = r"C:\Users\nisha\Desktop\2026 PS-II\Projects\VaR\pythonProject1\data"

# Isolate returns
delta = sp500_data['Return']

# Calculate 14-day average gains and losses
gain = delta.where(delta > 0, 0).rolling(window=14).mean()
loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()

# Calculate Relative Strength (RS)
rs = gain / loss

# Calculate RSI and scale to 0-1 range for ML input
sp500_data['RSI'] = (100 - (100 / (1 + rs))) / 100

# Drop the NaNs 
sp500_data.dropna(inplace=True)

# Safely construct the file path and export to CSV
export_file = os.path.join(path, 'Data_with_features.csv')
sp500_data.to_csv(export_file)

print(sp500_data)

                  Close         High          Low         Open      Volume  \
Date                                                                         
2001-05-16  1284.989990  1286.390015  1243.020020  1249.439941  1405300000   
2001-05-17  1288.489990  1296.479980  1282.650024  1284.989990  1355600000   
2001-05-18  1291.959961  1292.060059  1281.150024  1288.489990  1130800000   
2001-05-21  1312.829956  1312.949951  1287.869995  1291.959961  1174900000   
2001-05-22  1309.380005  1315.930054  1306.890015  1312.829956  1260400000   
...                 ...          ...          ...          ...         ...   
2026-04-15  7022.950195  7026.240234  6967.129883  6978.169922  5278610000   
2026-04-16  7041.279785  7051.229980  7008.520020  7037.779785  5173650000   
2026-04-17  7126.060059  7147.520020  7074.549805  7074.549805  6145300000   
2026-04-20  7109.140137  7122.649902  7084.410156  7117.049805  4661130000   
2026-04-21  7064.009766  7137.270020  7050.200195  7122.640137  